In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..","python"]))

import numpy as np
from gridr.core.grid.grid_resampling import array_grid_resampling
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook # enables plot interface in J notebook
    output_notebook()

image = mandrill[0]
grid_dtype = np.float64
data_in_dtype = np.float64
data_out_dtype = np.float64
array_in = image.astype(data_in_dtype)

# Build the identity transformation grid used as the starting point in
# several tutorials.
x = np.arange(0, image.shape[0], dtype=grid_dtype)
y = np.arange(0, image.shape[1], dtype=grid_dtype)
xx, yy = np.meshgrid(x, y)

# Support policy: which input samples must be valid

Each interpolated output reads a small neighbourhood of input samples. When some of them are masked, or lie outside the source array, GridR must decide whether the output is still valid. The *support policy* makes this decision explicit, with the `support_policy` parameter of `array_grid_resampling`.

**What you'll learn**

- What the stencil of an interpolator is, and why some of its weights are exactly zero
- The two support policies: `effective` (default) and `full`
- Their effect around masked input samples
- Their effect at the borders of the source array
- How to choose a policy, and what it does not change

This page builds on **Masking inputs**.

## Stencil and effective support

A separable interpolator combines K samples along each axis, K x K samples in all: its *stencil*.

| Interpolator | Stencil size K | Non-zero weights per axis at an integer coordinate |
|---|---|---|
| `nearest` | 1 | 1 |
| `linear` | 2 | 1, the sample itself |
| `cubic` | 4 | 1, the sample itself |
| `bspline<n>` (n odd) | n + 1 | n |

Away from integer coordinates, every weight of the stencil is non-zero. When the target coordinate is an integer along an axis, some weights of that axis are exactly zero.
For the linear and cubic interpolators, only the weight of the sample at that coordinate remains, equal to 1: the interpolation returns the input sample itself.
A B-spline of degree n keeps n non-zero weights out of n + 1.

The *effective support* is the set of samples with a non-zero weight. It is smaller than the stencil only at integer coordinates.

## The two policies

- **`effective`** (default): only the samples of the effective support must lie inside the source array and be valid in the input mask.
- **`full`**: every sample of the stencil must lie inside the source array and be valid, whatever its weight.

The policy covers both causes of invalidity: masked samples and samples outside the source array. Where both policies give a valid output, they give the same value. For `nearest`, the stencil is a single sample: both policies are identical.

## Around a masked sample

Let's mask one input sample and resample its neighbourhood on a fine grid, with a step of 0.1 pixel, so that some targets fall on integer coordinates.

In [ ]:
masked_pos = (60, 160)
array_in_mask = np.ones(array_in.shape, dtype=np.uint8)
array_in_mask[masked_pos] = 0

offsets = np.linspace(-3, 3, 61)  # step 0.1
xx, yy = np.meshgrid(masked_pos[1] + offsets, masked_pos[0] + offsets)

results = {}
for policy in ("effective", "full"):
    results[policy] = array_grid_resampling(
        interp="cubic",
        array_in=array_in,
        grid_row=yy,
        grid_col=xx,
        grid_resolution=(1, 1),
        array_in_mask=array_in_mask,
        array_out=None,
        array_out_mask=True,
        nodata_out=0,
        trust_padding=False,
        support_policy=policy,
    )
mask_effective = results["effective"][1]
mask_full = results["full"][1]

In [ ]:
import matplotlib.pyplot as plt

@mpl_plot_wrapper
def policy_scatter_plot(xx, yy, masks, prefix=None):
    fig, axes = plt.subplots(1, len(masks), figsize=(12, 6), sharey=True)
    for ax, (title, mask_out) in zip(axes, masks.items()):
        valid = mask_out == 1
        ax.scatter(xx[valid], yy[valid], marker='o', s=8, c='tab:blue', edgecolor='black',
                   linewidth=0.2, alpha=0.5, label='valid')
        ax.scatter(xx[~valid], yy[~valid], marker='o', s=8, c='tab:orange', edgecolor='black',
                   linewidth=0.5, alpha=0.9, label='invalid')
        ax.plot(masked_pos[1], masked_pos[0], 'rx', markersize=10, label='masked sample')
        ax.set_title(f'support_policy="{title}"')
        ax.set_xlabel('column')
        ax.set_aspect('equal')
    # Rows increase downwards, as in an image (the y axis is shared)
    axes[0].invert_yaxis()
    axes[0].set_ylabel('row')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=3)
    fig.subplots_adjust(bottom=0.15)
    return fig

In [ ]:
_ = policy_scatter_plot(xx, yy, {'effective': mask_effective, 'full': mask_full},
                    prefix='005_policy_masked_sample')

In [ ]:
for policy, (_, mask_out) in results.items():
    invalid = mask_out == 0
    print(f"{policy:>9}: {invalid.sum():4d} invalid outputs, "
          f"rows {yy[invalid].min():.1f} to {yy[invalid].max():.1f}, "
          f"columns {xx[invalid].min():.1f} to {xx[invalid].max():.1f}")

both = (mask_effective == 1) & (mask_full == 1)
print("Same values where both are valid:",
      np.array_equal(results["effective"][0][both], results["full"][0][both]))

Under `full`, an output is invalid as soon as the stencil contains the masked sample. The cubic stencil of a coordinate `y` covers the rows `floor(y) - 1` to `floor(y) + 2`, so it contains row 60 for `y` in [58, 62). The invalid area is the square [58, 62) x [158, 162): its shape depends only on the stencil, not on the position of the target within a pixel. Note that it includes the coordinate 58.0 itself: the stencil reaches row 60 there, even with a zero weight.

Under `effective`, the invalid area is the open square (58, 62) x (158, 162), crossed by valid lines at the integer coordinates 59 and 61. On the row 59, for instance, only the samples of row 59 have a non-zero weight, and none of them is masked. Row 60 and column 160 stay invalid, since their active samples include the masked one, except where they cross the valid lines.

## At the borders of the source array

The same rule applies to the samples that lie outside the source array. Let's zoom 4x into a small crop of the image.

In [ ]:
win_center = np.array((58, 175))  # left eye
win_shape = np.array((80, 80))
win = np.array((win_center - win_shape // 2,
                win_center - win_shape // 2 + win_shape - 1)).T
array_in_cropped = np.ascontiguousarray(
    array_in[win[0][0]:win[0][1]+1, win[1][0]:win[1][1]+1]
)

zoom = 4
x = np.arange(0, array_in_cropped.shape[0], dtype=grid_dtype)
y = np.arange(0, array_in_cropped.shape[1], dtype=grid_dtype)
xx, yy = np.meshgrid(x, y)

def border_masks(interp, **kwargs):
    masks = {}
    for policy in ("effective", "full"):
        _, masks[policy] = array_grid_resampling(
            interp=interp,
            array_in=array_in_cropped,
            grid_row=yy,
            grid_col=xx,
            grid_resolution=(zoom, zoom),
            array_out=None,
            array_out_mask=True,
            nodata_out=None,
            trust_padding=False,
            support_policy=policy,
            **kwargs,
        )
    return masks

cubic_masks = border_masks("cubic")

In [ ]:
# Output column k targets the source column k / 4
positions = np.arange(cubic_masks["effective"].shape[1]) / zoom
print("source column :", positions[:6], "...", positions[-6:])
for policy, mask_out in cubic_masks.items():
    print(f"{policy:>13} :", mask_out[40, :6], "...", mask_out[40, -6:])

Under `effective`, the outputs at the source columns 0.0 and 79.0 are valid: they return the samples themselves. Only the fractional positions within one pixel of the border are invalid, because their stencil needs a sample outside the array with a non-zero weight.

Under `full`, the whole stencil, columns `floor(x) - 1` to `floor(x) + 2`, must lie inside the array: the valid outputs are those with 1 <= x < 78. The invalid band is asymmetric, four positions on the left (0.0 to 0.75) and five on the right (78.0 to 79.0), because the cubic stencil has an even size.

In [ ]:
plot_im({'effective, top-left': cubic_masks["effective"][:24, :24],
         'full, top-left': cubic_masks["full"][:24, :24],
         'effective, bottom-right': cubic_masks["effective"][-24:, -24:],
         'full, bottom-right': cubic_masks["full"][-24:, -24:]},
        prefix='005_policy_border_corners')

In these corner views, black marks the invalid outputs. The linear interpolator gives the same picture at a smaller scale: its stencil covers `floor(x)` and `floor(x) + 1`.

In [ ]:
linear_masks = border_masks("linear")
for policy, mask_out in linear_masks.items():
    print(f"{policy:>9}: {(mask_out == 0).sum():4d} invalid outputs, "
          f"last columns of row 40: {mask_out[40, -3:]}")

Under `effective`, every output is valid. Under `full`, only the last row and the last column are invalid: their stencil reaches the sample 80, outside the array.

## Choosing a policy

**`effective`** (default) keeps every output whose non-zero weights read valid samples:

- it gives the largest valid output domain;
- at integer coordinates, it returns the input samples exactly, even next to a masked sample or to the border of the source array;
- it is the behaviour of the previous GridR versions (<0.8.0).

**`full`** decides the validity from the stencil alone:

- the invalid area around an invalid sample, or along a border, has a fixed shape: it does not depend on the position of the target within a pixel, and has no valid lines or points inside;
- the output mask stays consistent when the grid moves by a fraction of a pixel, which matters when it feeds further processing (filtering, morphology, mosaicking);
- it is slightly faster with masked data, since its validity test always covers the same window.

## What the support policy does not change

- **Grid masks.** With an oversampled grid, a masked grid node invalidates the positions where its bilinear weight is non-zero, whatever the support policy (see **Masking inputs**).
- **B-spline prefiltering.** For B-spline interpolators, the input mask is first dilated by the prefiltering (see **B-Spline masking**). The support policy then applies to the dilated mask.
- **Finite input values.** Under both policies, a masked sample can be read with a zero weight. The input array must hold finite values everywhere.